# YOLOv8 Training & Evaluation Pipeline 
## **FOR GOOGLE COLAB ONLY**

This notebook is designed as a standardized environment for testing and evaluating the YOLO models in Google Colab. 

Current hyperparameters are set to standard literature-based values for initial testing. These are not yet fully optimized for this specific dataset and serve as a simple baseline.

# SETUP

## Clone repo

In [ ]:
import os

REPO_URL = "https://github.com/kenami0981/DermaAI.git"
REPO_NAME = "DermaAI"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
%cd {REPO_NAME}
!ls

## Set up the Conda environment

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

In [ ]:
YAML_PATH = "Models/environment.yml"

!conda env create -f DermaAI/Models/environment.yml # it will take few minutes

In [ ]:
!conda run -n acne-detection python --version

In [ ]:
# !conda run -n acne-detection python main.py # jeśli by tak odpalać trening z pliku .py, nie trzeba wtedy ustawiać ścieżek

## Download the dataset

In [ ]:
# 1. Upload your dataset .zip to the "acne-data" folder on your Google Drive.
# (If you use a different folder, remember to update the path)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Assuming your file is named "Acne.v21i.yolov8.zip"
# (if not, make sure to change the name in the path below):

!cp /content/drive/MyDrive/acne-data/Acne.v21i.yolov8.zip /content/

In [ ]:
!unzip -q /content/Acne.v21i.yolov8.zip -d /content/DermaAI/Models/data

In [ ]:
# Uncomment if you want to unmount your Drive, you can configure it so that progress is saved there safetly 

# drive.flush_and_unmount()

# PATHS SETUP

In [ ]:
import sys
import os

env_path = "/usr/local/envs/acne-detection/lib/python3.10/site-packages"

if os.path.exists(env_path):
    sys.path.append(env_path)
    print("Conda environment successfully linked to the notebook.")
else:
    print(f"ERROR: Environment path not found: {env_path}")

## Quick test to see if everything is ready (RUN IT BEFORE HEAVY TRAINING!)



In [ ]:
from pathlib import Path
from ultralytics import YOLO

In [ ]:
ROOT = Path('/content/DermaAI/Models').resolve()
DATA_DIR = ROOT / "data"
DATA_YAML = DATA_DIR / "data.yaml"
MODELS_DIR = ROOT
YOLO_WEIGHTS = MODELS_DIR / "models" / "yolov8m.pt"
RUNS_DIR = ROOT / "runs"

print("FOLDER STRUCTURE VERIFICATION")
for folder in [DATA_DIR, MODELS_DIR / "models", RUNS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"Verified: {folder}")

if DATA_YAML.exists():
    print(f"Data configuration file found: {DATA_YAML}")
else:
    print(f"MISSING FILE: {DATA_YAML}. Ensure that the dataset is extracted to this directory!")

print("\nTEST RUN (1 EPOCH)")
try:
    model = YOLO("yolov8n.pt")
    model.train(
        data=str(DATA_YAML),
        project=str(RUNS_DIR),
        name="test_run",
        epochs=1,
        imgsz=320,
        device=0,
        exist_ok=True
    )

    EXPECTED_BEST = RUNS_DIR / "test_run" / "weights" / "best.pt"
    if EXPECTED_BEST.exists():
        print(f"\nSUCCESS! Model saved at: {EXPECTED_BEST}")
    else:
        print(f"Unexpected result: Training completed but weights not found at {EXPECTED_BEST}")

except Exception as e:
    print(f"Test failed with error: {e}")

# TRAINING

### IMPORTANT! Make sure the T4 GPU runtime is enabled
This script requires a GPU for efficient training. Go to Runtime (Środowisko wykonawcze) --> Change runtime type (Zmień typ środowiska wykonawczego) and select T4 GPU

### EVEN MORE IMPORTANT:

info for Colab Free (T4 GPU)
* **Max Lifetime:** Up to **12 hours** (but rarely guaranteed).
* **Idle Timeout:** **30–90 minutes**. If you stop interacting or close the tab, the session dies quickly.
* **Availability:** Google can reclaim the GPU at any time if demand is high.
* **Storage:** All local files are **deleted** once the session ends. Make sure to save everything on google drive or your device


In [ ]:
import os
import yaml
import pandas as pd
import zoneinfo

from pathlib import Path
from ultralytics import YOLO
from datetime import datetime

In [ ]:
# PATH CONFIGURATION 
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path('/content/DermaAI/Models').resolve()

DATA_DIR = ROOT / "data"
DATA_YAML = DATA_DIR / "data.yaml"
MODELS_DIR = ROOT / "models"
RUNS_DIR = ROOT / "runs"

# Ensure directories exist
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

YOLO_WEIGHTS = MODELS_DIR / "yolov8m.pt"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
def train_production():
    """
    Executes the training run with Colab settings and saves hyperparameters.
    """

    if not DATA_YAML.exists():
        print(f"ERROR: Configuration file not found at {DATA_YAML}")
        print("Ensure data is extracted to 'Models/data/' directory.")
        return
    
    pl_timezone = zoneinfo.ZoneInfo("Europe/Warsaw")
    timestamp = datetime.now(pl_timezone).strftime("%Y%m%d_%H%M")
    run_name = f"acne_yolo_train_colab_{timestamp}"

    print(f"Initializing Production Training...")
    print(f"Root: {ROOT}")
    print(f"Data: {DATA_YAML}")

    # Load model
    model = YOLO(str(YOLO_WEIGHTS))

    # Define training parameters in a dictionary for saving
    train_params = {
        # File System
        "data": str(DATA_YAML),
        "project": str(RUNS_DIR),
        "name": run_name,
        "exist_ok": True,
        "plots": True,
        "verbose": True,

        # Infrastructure (for Colab)
        "device": 0,           # Force GPU
        "workers": 4,          # Parallel data loading
        "batch": -1,           # AutoBatch

        # Training Strategy
        "epochs": 150,         # Extended training for convergence
        "imgsz": 800,          # High resolution for small skin lesions
        "patience": 30,        # Early stopping if mAP plateaus (important!)
        "save": True,
        "save_period": 5,      # Save checkpoint every 5 epochs

        # Optimization 
        "optimizer": 'AdamW',
        "lr0": 0.001,          # Initial learning rate
        "cos_lr": True,        # Cosine learning rate scheduler

        # Augmentation 
        "mosaic": 1.0,         # Combine 4 images to handle scale variance
        "mixup": 0.1,          # Blend images to reduce overfitting
        "hsv_h": 0.015,        # Adjust Hue (skin tone variations)
        "hsv_s": 0.5,          # Adjust Saturation (redness of inflammation)
        "hsv_v": 0.4,          # Adjust Value (shadows and bathroom lighting)
        "fliplr": 0.5,         # Horizontal flip
        "scale": 0.5,          # Zoom in/out to simulate camera distance
    }

    # START TRAINING
    model.train(**train_params)

    # SAVE HYPERPARAMETERS 
    # Define path to the weights folder of this specific run
    weights_dir = RUNS_DIR / train_params["name"] / "weights"
    weights_dir.mkdir(parents=True, exist_ok=True)
    
    config_save_path = weights_dir / "training_params.yaml"
    
    with open(config_save_path, 'w') as f:
        yaml.dump(train_params, f, default_flow_style=False)

    print(f"\nTraining Complete.")
    print(f"Results and config saved to: {weights_dir}")

if __name__ == "__main__":
    train_production()

# MODEL EVALUATION

In [ ]:
def get_metrics_from_csv():
    """
    Extracts the best performance metrics from the results.csv file
    """
    run_name = "test_run" # Change the name to the one you want to evaluate
    csv_path = RUNS_DIR / run_name / "results.csv"

    if not csv_path.exists():
        print(f"ERROR: results.csv not found at {csv_path}")
        return

    # Load the CSV file
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    # Find the row with the highest mAP50
    best_epoch = df.loc[df['metrics/mAP50(B)'].idxmax()]

    # Compile the key metrics
    results = {
        "Metric": [
            "Best Epoch",
            "mAP50",
            "mAP50-95",
            "Precision",
            "Recall"
        ],
        "Result": [
            int(best_epoch['epoch']),
            f"{best_epoch['metrics/mAP50(B) Lennox']:.4f}" if 'metrics/mAP50(B) Lennox' in best_epoch else f"{best_epoch['metrics/mAP50(B)']:.4f}",
            f"{best_epoch['metrics/mAP50-95(B)']:.4f}",
            f"{best_epoch['metrics/precision(B)']:.4f}",
            f"{best_epoch['metrics/recall(B)']:.4f}"
        ]
    }

    # Display the table
    metrics_df = pd.DataFrame(results)
    print("\n" + "="*40)
    print("BEST METRICS FROM TRAINING HISTORY (CSV)")
    print("="*40)
    print(metrics_df.to_string(index=False))
    print("="*40)

if __name__ == "__main__":
    get_metrics_from_csv()

In [ ]:
def show_per_class_metrics():
    """
    Displays detailed performance metrics for each individual class
    """
    run_name = "test_run" # Change the name to the one you want to evaluate
    model_path = RUNS_DIR / run_name / "weights" / "best.pt"

    if not model_path.exists():
        print(f"ERROR: Model not found at {model_path}")
        return

    # Load model and run validation
    model = YOLO(str(model_path))
    metrics = model.val(data=str(DATA_YAML), verbose=False, project=str(RUNS_DIR))

    # Get class names
    class_names = model.names

    per_class_data = []

    # metrics.box is an object containing arrays for each metric
    for i, name in class_names.items():
        per_class_data.append({
            "ID": i,
            "Class Name": name,
            "mAP50": f"{metrics.box.ap50[i]:.4f}",
            "mAP50-95": f"{metrics.box.ap[i]:.4f}",
            "Precision": f"{metrics.box.p[i]:.4f}",
            "Recall": f"{metrics.box.r[i]:.4f}"
        })

    # Create and display the table
    df = pd.DataFrame(per_class_data)

    print("\n" + "="*50)
    print("PERFORMANCE BY CLASS")
    print("="*50)
    print(df.to_string(index=False))
    print("="*50)

if __name__ == "__main__":
    show_per_class_metrics()

### **EVALUATION GUIDE**

| Term | Description | Simple Interpretation |
| :--- | :--- | :--- |
| **Precision** | Accuracy of detections | How many detected items were actually correct? |
| **Recall** | Ability to find objects | How many of the total objects did the model find? |
| **mAP50** | Mean Average Precision | Main metric. Success rate at 50% overlap (IoU). |
| **mAP50-95** | Strict mAP | Measures how perfectly the boxes fit the objects. |

### **YOLO Model Benchmarks**

| Metric | Baseline (Poor) | Target (Good) | Perfect |
| :--- | :--- | :--- | :--- |
| **mAP50** | < 0.30 | 0.50 - 0.70 | > 0.85 |
| **mAP50-95** | < 0.15 | 0.30 - 0.45 | > 0.60 |
| **Precision** | < 0.40 | 0.70 - 0.80 | > 0.90 |
| **Recall** | < 0.30 | 0.60 - 0.75 | > 0.85 |


**FYI:** These are the initial baseline metrics:

| ID | Class Name | mAP50 | mAP50-95 | Precision | Recall |
|:---|:---|:---|:---|:---|:---|
| 0 | **blackheads** | 0.0257 | 0.0077 | 0.1464 | 0.0278 |
| 1 | **dark spot** | 0.1852 | 0.0611 | 0.2225 | 0.3308 |
| 2 | **nodules** | 0.1166 | 0.0437 | 0.3067 | 0.1190 |
| 3 | **papules** | 0.2328 | 0.0708 | 0.3285 | 0.3433 |
| 4 | **pustules** | 0.2483 | 0.0825 | 0.2802 | 0.3292 |
| 5 | **whiteheads** | 0.1013 | 0.0300 | 0.1652 | 0.2128 |

Current performance is terrible, but good enough as a starting point.

# SAVE MODEL AND WEIGHTS

Save it where you see fit


In [ ]:
drive.mount('/content/drive')

In [ ]:
!ls

In [ ]:
!cp -r /content/DermaAI/Models/runs/acne_yolo_train_colab /content/drive/MyDrive/acne-data/
!cp -r /content/DermaAI/Models/models /content/drive/MyDrive/acne-data/

In [ ]:
drive.flush_and_unmount()